# Pipeline B — 19 yes/no questions + reasoning

Same idea as Tousi et al. (2025), Pipeline B, with **Gemma 4 26B** (our best model from Pipeline A):

1. **Ask:** the vision model looks at each collage and answers **19 yes/no questions** (the paper's questions) in one call.
2. **Combine:** the same model, now *without* the image, reads the 19 answers and decides how likely a gully is (score 1–5), with a short reason.

The 19 answers are also saved as 0/1 columns: they are the input features for **Pipeline C** (the small classifier).

Runs on all 621 Missouri + 300 Hisar collages. About 3 hours.

**Settings (right panel):** GPU T4 x2 · Internet On · Input `gully-collages` (must contain collages_missouri and collages_hisar).
**Run with:** Save Version → **Save & Run All (Commit)**. Files are in the version's **Output** tab:
`pipeline_b_answers_{missouri,hisar}.csv`, `pipeline_b_decisions_{missouri,hisar}.csv`, `pipeline_b_summary.csv`, `pipeline_b_question_stats.csv`.

**If a run gets cut off:** Add Input → Your Work → this notebook's last version → Save & Run All again; finished rows are reused.

In [ ]:
# 1. GPU check
!nvidia-smi

In [ ]:
# 2. Install Ollama
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 3. Start Ollama server
import subprocess, time, requests
subprocess.Popen("ollama serve > /kaggle/working/ollama.log 2>&1", shell=True)
for _ in range(60):
    try:
        requests.get("http://localhost:11434", timeout=2); print("Ollama server is running"); break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("Ollama did not start - see /kaggle/working/ollama.log")
!ollama --version

In [ ]:
# 4. Settings, data, checkpoint restore
import base64, csv, glob, json, os, re, shutil
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

MODEL = "gemma4:26b"
DATASETS = ["missouri", "hisar"]
TEST_PER_CLASS = 75          # the same 150 Missouri fields used to choose the model in Pipeline A
OLLAMA = "http://localhost:11434"
OUT_DIR = "/kaggle/working"

for src in glob.glob("/kaggle/input/**/pipeline_b_*.csv", recursive=True):
    dst = os.path.join(OUT_DIR, os.path.basename(src))
    if len(pd.read_csv(src)) > (len(pd.read_csv(dst)) if os.path.exists(dst) else -1):
        shutil.copy(src, dst); print("Restored checkpoint:", os.path.basename(src))

def find_dataset(name):
    hits = [p for p in glob.glob("/kaggle/input/**/collages.csv", recursive=True) if name in p.lower()]
    if not hits:
        raise FileNotFoundError(f"No collages.csv for '{name}' - does gully-collages contain collages_{name}?")
    folder = os.path.dirname(sorted(hits, key=len)[0])
    df = pd.read_csv(folder + "/collages.csv", dtype={"label": str})
    df["path"] = folder + "/" + df["collage"]
    df = df[df["path"].map(os.path.exists)].reset_index(drop=True)
    print(f"{name}: {len(df)} collages")
    return df

data = {name: find_dataset(name) for name in DATASETS}
m = data["missouri"]
test_ids = set(m[m["label"].isin(["0", "1"])].groupby("label").sample(n=TEST_PER_CLASS, random_state=42)["tile_id"])
print("Missouri test subset:", len(test_ids), "fields (used only to choose the cut-off)")

In [ ]:
# 5. The 19 questions (Tousi et al. 2025, Section 4.2.2) and the two prompts
QUESTIONS = [
    "Do you see a low point in the terrain?",
    "Do narrow, winding paths or channels appear?",
    "Are there winding paths that become intermittent recurrent?",
    "Are there any linear depressions or ruts which appear more pronounced along natural drainage lines or slopes?",
    "Are there narrow and shallow channels which appear intermittently deeper or more indented into the soil?",
    "Are there areas where soil appears disturbed or vegetation is removed?",
    "Does a specific path lack vegetation, suggesting an evolving or emerging channel?",
    "Are there varying types and levels of coarseness in the texture of the soil?",
    "Are there clear starting and ending points of potential channels?",
    "Are there small rills or grooves indicating water flow?",
    "Is there a varying exposure of lighter or darker colored soil?",
    "Are there sediment accumulations forming?",
    # Q13 is cut off in the published paper; completed here with "areas across the images?"
    "Are there signs of water activity, like soil clumps or crusting, that appear or intensify in specific areas across the images?",
    "Are there any branching patterns that resemble temporary streams?",
    "Are there indications of nearby human activity, such as tillage or machinery tracks?",
    "Do you see any sign of water flow patterns across the field in multiple images?",
    "Do you see any edges in the images indicating removal of soil along the water pathway?",
    "Do you see any cuts in the soil associated with water flow across the field?",
    "Do you see any indication of human activity such as tillage that is not naturally happened in the field?",
]
QCOLS = [f"q{i+1}" for i in range(len(QUESTIONS))]

ASK_PROMPT = ("Given this collage of {n} images of the exact same agricultural area, collected over several years "
              "(each tile is labelled with its date or image number; ignore any grey 'no image' tile or text tile), "
              "first describe in one or two sentences what you see, then answer each question with yes or no.\n\n"
              + "\n".join(f"{c}. {q}" for c, q in zip(QCOLS, QUESTIONS)))
ASK_SCHEMA = {"type": "object",
              "properties": {"description": {"type": "string"},
                             **{c: {"type": "string", "enum": ["yes", "no"]} for c in QCOLS}},
              "required": ["description"] + QCOLS}

DECIDE_PROMPT = ("A vision model examined a collage of overhead images of one agricultural field, taken in different years, "
                 "and answered yes/no questions about it.\n\n"
                 "Its description: {description}\n\nIts answers:\n{qa}\n\n"
                 "An ephemeral gully is a small erosion channel cut by rainwater runoff, which farmers fill in during "
                 "tillage and which tends to reappear in the same place. Many fields have no gully.\n"
                 "Based only on these answers, reason briefly about whether this field contains an ephemeral gully, "
                 "then score how likely it is: 1 = almost certainly no, 2 = probably no, 3 = unsure, "
                 "4 = probably yes, 5 = almost certainly yes.")
DECIDE_SCHEMA = {"type": "object",
                 "properties": {"reasoning": {"type": "string"},
                                "score": {"type": "integer", "enum": [1, 2, 3, 4, 5]}},
                 "required": ["reasoning", "score"]}
print(len(QUESTIONS), "questions")

In [ ]:
# 6. Helpers: model download, robust chat call, resumable runner
def pull(model):
    print(f"Downloading {model} ...", flush=True)
    r = requests.post(f"{OLLAMA}/api/pull", json={"model": model, "stream": False}, timeout=3600)
    r.raise_for_status(); print("  done:", r.json().get("status"))

def chat(content, schema, image_path=None):
    msg = {"role": "user", "content": content}
    if image_path:
        with open(image_path, "rb") as f:
            msg["images"] = [base64.b64encode(f.read()).decode()]
    body = {"model": MODEL, "messages": [msg], "format": schema, "stream": False, "think": False,
            "options": {"temperature": 0, "num_ctx": 8192, "seed": 42, "num_predict": 900}}
    r = requests.post(f"{OLLAMA}/api/chat", json=body, timeout=900)
    if r.status_code == 400 and "think" in r.text.lower():
        body.pop("think"); r = requests.post(f"{OLLAMA}/api/chat", json=body, timeout=900)
    r.raise_for_status()
    return r.json()["message"]["content"]

def good_rows(p):
    d = pd.read_csv(p, dtype={"label": str})
    return d[d["error"].isna() | (d["error"].astype(str).str.strip() == "")]

def run(name, df, out_csv, columns, work):
    # work(row) -> dict with the columns; failed rows are retried on the next run
    if os.path.exists(out_csv):
        good_rows(out_csv).to_csv(out_csv, index=False)
    done = set(pd.read_csv(out_csv)["tile_id"]) if os.path.exists(out_csv) else set()
    todo = df[~df["tile_id"].isin(done)]
    new = not os.path.exists(out_csv)
    with open(out_csv, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["tile_id", "label"] + columns + ["seconds", "error"])
        if new:
            w.writeheader()
        errors_in_a_row = 0
        for _, row in tqdm(todo.iterrows(), total=len(todo), desc=os.path.basename(out_csv)):
            t = time.time()
            rec = {"tile_id": row["tile_id"], "label": row["label"] if pd.notna(row["label"]) else ""}
            try:
                rec.update(work(row)); rec["error"] = ""; errors_in_a_row = 0
            except Exception as e:
                rec["error"] = f"{type(e).__name__}: {e}"[:300]; errors_in_a_row += 1
            rec["seconds"] = round(time.time() - t, 1)
            w.writerow(rec); f.flush()
            if errors_in_a_row >= 10:
                raise RuntimeError(f"10 errors in a row in {name}, last: {rec['error']}")
    return pd.read_csv(out_csv, dtype={"label": str})

pull(MODEL)

In [ ]:
# 7. Stage 1 - ask the 19 questions (vision)
def ask_work(row):
    raw = chat(ASK_PROMPT.format(n=row["n_images"]), ASK_SCHEMA, image_path=row["path"])
    out = json.loads(raw)
    rec = {c: (1 if str(out.get(c, "")).strip().lower() == "yes" else 0) for c in QCOLS}
    rec["description"] = str(out.get("description", ""))
    return rec

answers = {name: run(name, df, f"{OUT_DIR}/pipeline_b_answers_{name}.csv", QCOLS + ["description"], ask_work)
           for name, df in data.items()}

In [ ]:
# 8. Stage 2 - combine the 19 answers into one decision (text only, no image)
def decide_work_for(ans_df):
    lookup = ans_df.set_index("tile_id")
    def work(row):
        a = lookup.loc[row["tile_id"]]
        qa = "\n".join(f"- {q} {'Yes' if int(a[c]) == 1 else 'No'}" for c, q in zip(QCOLS, QUESTIONS))
        raw = chat(DECIDE_PROMPT.format(description=a["description"], qa=qa), DECIDE_SCHEMA)
        out = json.loads(raw)
        return {"score": int(out["score"]), "reasoning": str(out.get("reasoning", ""))}
    return work

decisions = {}
for name, df in data.items():
    ok = good_rows(f"{OUT_DIR}/pipeline_b_answers_{name}.csv")
    decisions[name] = run(name, df[df["tile_id"].isin(ok["tile_id"])],
                          f"{OUT_DIR}/pipeline_b_decisions_{name}.csv", ["score", "reasoning"],
                          decide_work_for(ok))

In [ ]:
# 9. Results
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score, f1_score, confusion_matrix

def metrics(d, thr):
    y, s = d["label"].astype(int), d["score"].astype(int)
    p = (s >= thr).astype(int)
    return {"n": len(d), "AUC": round(roc_auc_score(y, s), 3), "threshold": thr,
            "balanced_acc": round(balanced_accuracy_score(y, p), 3), "acc": round(accuracy_score(y, p), 3),
            "F1": round(f1_score(y, p, zero_division=0), 3), "yes_rate": round(p.mean(), 2),
            "confusion": confusion_matrix(y, p, labels=[0, 1]).tolist()}

md_ = good_rows(f"{OUT_DIR}/pipeline_b_decisions_missouri.csv")
md_ = md_[md_["label"].isin(["0", "1"])]
test, held = md_[md_["tile_id"].isin(test_ids)], md_[~md_["tile_id"].isin(test_ids)]
thr = max([2, 3, 4, 5], key=lambda t: balanced_accuracy_score(test["label"].astype(int), (test["score"].astype(int) >= t)))
summary = [{"dataset": "missouri_test150", **metrics(test, thr)},
           {"dataset": "missouri_heldout", **metrics(held, thr)},
           {"dataset": "missouri_all", **metrics(md_, thr)}]
hd = good_rows(f"{OUT_DIR}/pipeline_b_decisions_hisar.csv")
hs = hd["score"].astype(int)
summary.append({"dataset": "hisar", "n": len(hd), "threshold": thr, "yes_rate": round((hs >= thr).mean(), 2),
                "score_counts": hs.value_counts().sort_index().to_dict()})
pd.DataFrame(summary).to_csv(f"{OUT_DIR}/pipeline_b_summary.csv", index=False)
print(pd.DataFrame(summary).to_string(index=False))
print("\nPipeline A (Gemma 4 26B), Missouri held-out 471: AUC 0.862, balanced acc 0.817")

# how often each question is answered 'yes' on gully vs no-gully Missouri fields (which questions carry signal)
ma = good_rows(f"{OUT_DIR}/pipeline_b_answers_missouri.csv"); ma = ma[ma["label"].isin(["0", "1"])]
ha = good_rows(f"{OUT_DIR}/pipeline_b_answers_hisar.csv")
qs = pd.DataFrame({"question": QUESTIONS,
                   "yes_missouri_gully": ma[ma.label == "1"][QCOLS].mean().round(2).values,
                   "yes_missouri_no_gully": ma[ma.label == "0"][QCOLS].mean().round(2).values,
                   "yes_hisar": ha[QCOLS].mean().round(2).values}, index=QCOLS)
qs["difference"] = (qs["yes_missouri_gully"] - qs["yes_missouri_no_gully"]).round(2)
qs.to_csv(f"{OUT_DIR}/pipeline_b_question_stats.csv")
pd.set_option("display.max_colwidth", 70)
print("\n", qs.sort_values("difference", ascending=False).to_string())